# Phase 1 - Detection head  —  Rotax_916_iS

Binary: is anything wrong right now? In v3 the label fires on **either** a sensor
fault **or** an engine failure mode, so this head covers both families.

Trains the shared TCN encoder from scratch; every later phase warm-starts from it.

In [1]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath("."))
import numpy as np, tensorflow as tf
from tensorflow import keras
import tf_data_pipeline as P, model_architectures as A, train_common as C

C.report_gpu()

  TF 2.16.2  devices ['CPU', 'GPU']


['CPU', 'GPU']

In [2]:
# ---- SET THE ENGINE HERE. Run this notebook once per engine. ----
ENGINE = "Rotax_916_iS"
steps = C.describe(ENGINE)
train_ds, val_ds, test_ds = C.datasets(ENGINE)

Rotax_916_iS  (916)
  physics   : v3   TBO 1200.0 h
  rows      : 10,000,874 in 1922 scenarios
  steps/epoch: train 949  val 119  test 129
  window 128  stride 64  features 25  aux 10


2026-09-12 21:15:22.021788: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-09-12 21:15:22.022160: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-09-12 21:15:22.022176: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-09-12 21:15:22.022483: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-12 21:15:22.022506: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


### Model
TCN encoder (6 causal-dilated blocks, receptive field 253 > 128 window) + a small dense head.

In [3]:
xi  = A.build_encoder_input()
enc = A.build_encoder(xi)
out = A.build_detection_head(enc)
model = keras.Model(xi, {"y_detection": out}, name="detection")
model.summary()

Model: "detection"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x (InputLayer)      │ (None, 128, 25)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_conv1 (Conv1D) │ (None, 128, 48)   │      3,648 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_drop1          │ (None, 128, 48)   │          0 │ tcn0_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_conv2 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn0_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_drop2          │ (None, 128, 48)   │          0 │ tcn0_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_proj (Conv1D)  │ (None, 128, 48)   │      1,248 │ x[0][0]           │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_add (Add)      │ (None, 128, 48)   │          0 │ tcn0_drop2[0][0], │
│                     │                   │            │ tcn0_proj[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn0_relu (ReLU)    │ (None, 128, 48)   │          0 │ tcn0_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv1 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn0_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_drop1          │ (None, 128, 48)   │          0 │ tcn1_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_conv2 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn1_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_drop2          │ (None, 128, 48)   │          0 │ tcn1_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_add (Add)      │ (None, 128, 48)   │          0 │ tcn1_drop2[0][0], │
│                     │                   │            │ tcn0_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn1_relu (ReLU)    │ (None, 128, 48)   │          0 │ tcn1_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv1 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn1_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_drop1          │ (None, 128, 48)   │          0 │ tcn2_conv1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_conv2 (Conv1D) │ (None, 128, 48)   │      6,960 │ tcn2_drop1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_drop2          │ (None, 128, 48)   │          0 │ tcn2_conv2[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_add (Add)      │ (None, 128, 48)   │          0 │ tcn2_drop2[0][0], │
│                     │                   │            │ tcn1_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tcn2_relu (ReLU)    │ (None, 128, 48)   │          0 │ tcn2_add[0][0]    │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 86,129 (336.44 KB)

 Trainable params: 86,129 (336.44 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# (nothing extra for this phase)

In [5]:
model.compile(optimizer=keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss={"y_detection": keras.losses.BinaryCrossentropy()},
              metrics={"y_detection": [keras.metrics.BinaryAccuracy(name="acc"),
                                       keras.metrics.AUC(name="auc")]})

In [6]:
PHASE = "phase1_detection"
PREV  = None
ckpt  = C.ckpt_path(ENGINE, PHASE)
if PREV: C.warm_start(model, C.ckpt_path(ENGINE, PREV))

hist = model.fit(
    train_ds.map(C.split_labels(["y_detection"])),
    validation_data=val_ds.map(C.split_labels(["y_detection"])),
    steps_per_epoch=steps["train"],
    validation_steps=steps["val"],
    epochs=30,
    callbacks=C.callbacks(ckpt, monitor="val_loss", mode="min"),
    verbose=1,
)
print("saved:", ckpt)

Epoch 1/30


/Users/harsh/Documents/UAV_Engine/validation/venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-09-12 21:15:23.698179: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


949/949 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - acc: 0.9326 - auc: 0.9737 - loss: 0.1671
Epoch 1: val_loss improved from None to 0.11866, saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase1_detection_916.keras

Epoch 1: finished saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase1_detection_916.keras
949/949 ━━━━━━━━━━━━━━━━━━━━ 208s 212ms/step - acc: 0.9326 - auc: 0.9737 - loss: 0.1671 - val_acc: 0.9488 - val_auc: 0.9849 - val_loss: 0.1187 - learning_rate: 0.0010
Epoch 2/30
949/949 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step - acc: 0.9539 - auc: 0.9842 - loss: 0.1127
Epoch 2: val_loss improved from 0.11866 to 0.10696, saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase1_detection_916.keras

Epoch 2: finished saving model to /Users/harsh/Documents/UAV_Engine/validation/models/phase1_detection_916.keras
949/949 ━━━━━━━━━━━━━━━━━━━━ 214s 225ms/step - acc: 0.9539 - auc: 0.9842 - loss: 0.1127 - val_acc: 0.9566 - val_auc: 0.9870 - val_l

In [7]:
res = model.evaluate(test_ds.map(C.split_labels(["y_detection"])), steps=steps["test"], verbose=1, return_dict=True)
print(json.dumps({k: float(v) for k, v in res.items()}, indent=2))
with open(os.path.join(C.MODELS_DIR, f"{PHASE}_{C.ENGINE_KEY[ENGINE]}_test.json"), "w") as f:
    json.dump({k: float(v) for k, v in res.items()}, f, indent=2)

129/129 ━━━━━━━━━━━━━━━━━━━━ 22s 168ms/step - acc: 0.9721 - auc: 0.9876 - loss: 0.0862
{
  "acc": 0.9720808863639832,
  "auc": 0.9875583648681641,
  "loss": 0.08619792014360428
}
